# Starkregen im Central Park (NYC)
Gecko Green-Blue Infrastructure 2024

In [1]:
#setup system
%matplotlib inline
import numpy as np
import pandas as pd
import plotly.express as px


In [73]:
# read data
fi = '3653897.csv'
dummy = pd.read_csv(fi)

# make date a time stamp
dummy.DATE = pd.to_datetime(dummy.DATE)
dummy.index = dummy.DATE

# convert inch/day to mm/day
dummy['PRCPmm'] = dummy.PRCP*25.4


In [86]:
# plot data
px.scatter(dummy.PRCPmm, marginal_y='violin', template='none')#, marginal_x='box', marginal_y='box')

In [101]:
# simple histogram of data
px.histogram(dummy.PRCPmm, marginal="box", template='none',log_y=True)

In [102]:
dummy.PRCPmm.quantile(0.99)

43.687999999999995

In [106]:
#plot time series of 1% strongest rains
px.scatter(dummy.loc[dummy.PRCPmm>dummy.PRCPmm.quantile(0.99),'PRCPmm'], template='none', trendline="ols")


In [104]:

# plot counts of strongest rains per year
#dummy.PRCPmm.resample('1Y').max().sort_values()
datax = pd.DataFrame(np.unique(dummy.loc[dummy.PRCPmm>dummy.PRCPmm.quantile(0.99),'PRCPmm'].index.year.values, return_counts=True)).T
datax.columns = ['year', 'count']
datax['r_year'] = datax['year'] - datax['year'].min()
fig = px.scatter(datax, x='year', y='count', template='none', trendline="ols")
fig.show()

results = px.get_trendline_results(fig)
results.iloc[0]["px_fit_results"].summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.064
Model:                            OLS   Adj. R-squared:                  0.057
Method:                 Least Squares   F-statistic:                     9.302
Date:                Tue, 09 Apr 2024   Prob (F-statistic):            0.00275
Time:                        07:30:58   Log-Likelihood:                -293.08
No. Observations:                 138   AIC:                             590.2
Df Residuals:                     136   BIC:                             596.0
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        -20.3541      7.923     -2.569      0.011     -36.022      -4.686
x1             0.0124      0.004      3.050      0.003       0.004       0.020
==============================================================================
Omnibus:                       27.402   Durbin-Watson:                   1.658
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               41.639
Skew:                           1.001   Prob(JB):                     9.08e-10
Kurtosis:                       4.799   Cond. No.                     8.92e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 8.92e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [107]:
# plot sorted data
px.scatter(dummy.PRCPmm.sort_values().values, template='none')

# plot sorted data with undercut probability
#fig = px.scatter(x=(np.arange(len(dummy))+1)/(len(dummy)+1), y=dummy.PRCPmm.sort_values().values, template='none')
#fig.update_layout(xaxis_title='P_undercut', yaxis_title='Precip (mm/day)')
#fig.show()

In [90]:
annual_max = pd.DataFrame([dummy.PRCPmm.resample('1Y').max().sort_values().values, dummy.PRCPmm.resample('1Y').max().sort_values().index.year, (np.arange(145)+1)/146,(1/(1-(np.arange(145)+1)/146))]).T
annual_max.columns = ['Precip','Year','Probability','Annuality']
#px.scatter(annual_max, x='Probability', y='Precip', color='Year', template='none')
px.scatter(annual_max, x='Annuality', y='Precip', color='Year', template='none', log_x=True)


In [88]:
px.scatter(annual_max, x='Year', y='Precip', color='Year', template='none')

In [115]:
idx = annual_max.Year.sort_values().index
px.scatter(x=annual_max.loc[idx,'Year'].astype(int),y=annual_max.loc[idx,'Annuality'].rolling(25).mean().values, template='none')